# 2. Our pipeline: suffix SMC plus averaged self-normalized estimators

This is the main ARI experiment. Phase A generates an IID direct reference.
Phase B runs independent SMC components with different caps and terminal
`lambda_j`. For each component and threshold it computes a separately
normalized estimator `F_j` using the terminal SMC weights; the reported
SMC-SNIS estimate is the average of the `F_j` values. By default the final
SMC level is not resampled, avoiding an extra source of Monte Carlo variance.
Samples from different components are never pooled into one
importance ratio.

The final cells compare against direct sampling, report variance and relative
error, diagnose every lambda, and show boxplots of the tilted populations.


In [ ]:
# 1. Connect Drive, clone/update the repository, and install dependencies.
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import base64
import getpass
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main.git'
REPO_ROOT = Path('/content/rea_of_llms_supplementary_code-main')

def get_github_token():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
    if not token:
        token = getpass.getpass('GitHub token for private repo: ')
    if not token:
        raise RuntimeError('A GitHub token is required because the repository is private.')
    return token.strip()

GITHUB_TOKEN = get_github_token()
AUTH_HEADER = 'AUTHORIZATION: basic ' + base64.b64encode(
    f'x-access-token:{GITHUB_TOKEN}'.encode()
).decode()

def git_cmd(*args):
    return ['git', '-c', f'http.https://github.com/.extraheader={AUTH_HEADER}', *args]

def redact_cmd(cmd):
    return ['<github-auth-header>' if 'extraheader=' in str(part).lower() else str(part)
            for part in cmd]

def run(cmd, **kwargs):
    result = subprocess.run(cmd, text=True, capture_output=True, **kwargs)
    if result.returncode:
        print('Command failed:', ' '.join(redact_cmd(cmd)))
        if result.stdout:
            print('--- stdout ---')
            print(result.stdout)
        if result.stderr:
            print('--- stderr ---')
            print(result.stderr)
        raise subprocess.CalledProcessError(result.returncode, cmd, result.stdout, result.stderr)
    return result

if (REPO_ROOT / '.git').is_dir():
    run(git_cmd('-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'main'))
elif REPO_ROOT.exists():
    # Colab can leave a partial directory after an interrupted git clone.
    if str(REPO_ROOT).startswith('/content/'):
        shutil.rmtree(REPO_ROOT)
        run(git_cmd('clone', '--depth', '1', REPO_URL, str(REPO_ROOT)))
    else:
        raise RuntimeError(f'{REPO_ROOT} exists but is not a Git repository.')
else:
    run(git_cmd('clone', '--depth', '1', REPO_URL, str(REPO_ROOT)))

os.chdir(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Ready:', REPO_ROOT)


In [ ]:
# 2. EDIT PARAMETERS HERE.
# Phase A: direct=True, SMC=False. Phase B: direct=False, SMC=True.
# Phase C (analysis only): both False.
RUN_OFFICIAL_PUBLIC_DIRECT = False
RUN_OFFICIAL_PUBLIC_SMC = False
RUN_DRIVE_AUDIT = False  # Optional metadata-only inspection; generates nothing.
ARI_CONVENTION = 'official_public'

# Change this one path if your data lives in a different Drive folder.
OUTPUT_ROOT = Path('/content/drive/MyDrive/rea_official_public_validation')

COMPONENTS = [
    {'cap': 6,  'lambda': -0.5},
    {'cap': 6,  'lambda': -1.0},
    {'cap': 6,  'lambda': -1.5},
    {'cap': 7,  'lambda': -1.0},
    {'cap': 7,  'lambda': -1.5},
    {'cap': 7,  'lambda': -2.0},
    {'cap': 8,  'lambda': -1.5},
    {'cap': 8,  'lambda': -2.0},
    {'cap': 8,  'lambda': -2.5},
]
SEED_GROUPS = [500, 600, 700]
TAIL_THRESHOLDS = [6, 7, 8]
TAIL_THRESHOLDS_CSV = ','.join(map(str, TAIL_THRESHOLDS))
NUM_PARTICLES = 256
MCMC_STEPS_PER_LEVEL = 10
LAMBDA_STEP = 0.25
ESS_THRESHOLD = 0.5
RESAMPLE_FINAL = False  # True reproduces the older unweighted final population.
MAX_NEW_TOKENS = 100
SUFFIX_SIZES = '1,2,4,8,16,32,64,100'
SUFFIX_RHOS = '0.20,0.18,0.16,0.14,0.12,0.10,0.06,0.04'
BATCH_SIZE = 32
DEVICE = 'cuda'

OFFICIAL_PUBLIC_DIRECT_NUM_SAMPLES = 200_000
OFFICIAL_PUBLIC_DIRECT_BATCH_SIZE = 256
OFFICIAL_PUBLIC_DIRECT_SEED = 91_001

MODEL_NAME = 'roneneldan/TinyStories-8M'
TOKENIZER_NAME = 'EleutherAI/gpt-neo-125M'
PROMPT = 'Once upon a time, in a big forest, there lived a rhinoc'
TEMPERATURE = 1.0

assert ARI_CONVENTION == 'official_public'
assert TAIL_THRESHOLDS == [6, 7, 8] and TEMPERATURE == 1.0
print(f'Phase A - generate direct: {RUN_OFFICIAL_PUBLIC_DIRECT}')
print(f'Phase B - generate SMC:    {RUN_OFFICIAL_PUBLIC_SMC}')
print(f'Phase C - analysis only:   {not RUN_OFFICIAL_PUBLIC_DIRECT and not RUN_OFFICIAL_PUBLIC_SMC}')
print(f'Planned components:  {len(COMPONENTS) * len(SEED_GROUPS)}')
print(f'Final resampling:    {RESAMPLE_FINAL}')
print(f'Data folder:         {OUTPUT_ROOT}')


In [ ]:
# 3. Optional read-only audit of existing Drive artifacts. This cell never writes files.
import pandas as pd
from IPython.display import display
from rea_llms.data_reuse_audit import REUSE_DIMENSIONS, audit_run_directory

if RUN_DRIVE_AUDIT:
    audit_rows = audit_run_directory(
        OUTPUT_ROOT, target_ari_convention=ARI_CONVENTION,
    )
    if not audit_rows:
        print(f'No recognized run artifacts found under {OUTPUT_ROOT}')
    else:
        audit_df = pd.DataFrame(audit_rows)
        inventory_columns = [
            'artifact_type', 'algorithm', 'ari_convention', 'prompt_sha256',
            'model_name', 'tokenizer_name', 'max_new_tokens', 'ari_cap',
            'lambda_final', 'num_particles', 'mcmc_steps_per_level', 'seed',
            'has_raw_ari', 'has_final_smc_weights', 'has_log_z',
            'has_token_ids', 'runtime_seconds', 'generated_token_count',
            'proposal_count', *REUSE_DIMENSIONS, 'path',
        ]
        display(audit_df.reindex(columns=inventory_columns))
        failed = []
        for _, row in audit_df.iterrows():
            reasons = [row.get(f'reason_{key}') for key in REUSE_DIMENSIONS if not row.get(key, False)]
            failed.append('; '.join(reason for reason in reasons if isinstance(reason, str)))
        limitations = audit_df[['path']].copy()
        limitations['limitations'] = failed
        display(limitations)
        print(f'Read-only audit complete: {len(audit_df)} artifacts; no samples generated.')
else:
    print('Drive audit disabled. Set RUN_DRIVE_AUDIT=True to inspect existing artifacts.')


In [ ]:
# 4. Phase A: generate once, or load, the official-public IID direct reference.
import json
import torch

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('DEVICE=cuda but this Colab runtime has no GPU.')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DIRECT_ROOT = OUTPUT_ROOT / 'official_public_direct_reference'
if RUN_OFFICIAL_PUBLIC_DIRECT:
    command = [
        sys.executable, '-m', 'rea_llms.current_direct_reference',
        '--model-name', MODEL_NAME, '--tokenizer-name', TOKENIZER_NAME,
        '--prompt', PROMPT, '--device', DEVICE,
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--num-samples', str(OFFICIAL_PUBLIC_DIRECT_NUM_SAMPLES),
        '--batch-size', str(OFFICIAL_PUBLIC_DIRECT_BATCH_SIZE),
        '--seed', str(OFFICIAL_PUBLIC_DIRECT_SEED),
        '--thresholds', TAIL_THRESHOLDS_CSV,
        '--ari-convention', ARI_CONVENTION,
        '--output-dir', str(DIRECT_ROOT),
        '--skip-completed',
    ]
    subprocess.run(command, check=True)
direct_summary_path = DIRECT_ROOT / 'summary.json'
if not direct_summary_path.is_file():
    raise FileNotFoundError('Official-public direct cache missing. Run Phase A once.')
direct_summary = json.loads(direct_summary_path.read_text())
metadata = direct_summary['metadata']
assert metadata['ari_convention'] == ARI_CONVENTION
assert metadata['prompt_sha256'] == 'b4b7f60343cd3d1e24d8f2e312810afdaaa28034b9a194c66865b47483f70f59'
assert metadata['model_name'] == MODEL_NAME and metadata['tokenizer_name'] == TOKENIZER_NAME
assert metadata['max_new_tokens'] == MAX_NEW_TOKENS
assert 'decode the full prompt-token plus completion-token sequence' in metadata['decoding_convention']
assert direct_summary['runtime_seconds'] is not None
assert direct_summary['generated_token_count'] == OFFICIAL_PUBLIC_DIRECT_NUM_SAMPLES * MAX_NEW_TOKENS
print(f"Official-public direct ready: N={direct_summary['statistics']['count']:,}; runtime={direct_summary['runtime_seconds']/60:.1f} min")

In [ ]:
# 5. Phase B: generate missing official-public SMC components, or load exact caches.
# This cell does not write reports, tables, or plots to Drive.
from suffix_direct_mh_smc_experiment.run_tail_validation import parse_args, run_validation

arguments = [
    '--model-name', MODEL_NAME, '--tokenizer-name', TOKENIZER_NAME,
    '--prompt', PROMPT, '--device', DEVICE,
    '--components-json', json.dumps(COMPONENTS),
    '--seed-groups', ','.join(map(str, SEED_GROUPS)),
    '--thresholds', TAIL_THRESHOLDS_CSV, '--ari-convention', ARI_CONVENTION,
    '--lambda-step', str(LAMBDA_STEP),
    '--num-particles', str(NUM_PARTICLES), '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--mcmc-steps-per-level', str(MCMC_STEPS_PER_LEVEL),
    '--ess-threshold', str(ESS_THRESHOLD), '--suffix-sizes', SUFFIX_SIZES,
    '--suffix-rhos', SUFFIX_RHOS, '--batch-size', str(BATCH_SIZE),
    '--output-root', str(OUTPUT_ROOT),
    '--direct-summary', str(direct_summary_path), '--skip-completed-runs',
]
arguments.append('--resample-final' if RESAMPLE_FINAL else '--no-resample-final')
if not RUN_OFFICIAL_PUBLIC_SMC:
    arguments.append('--audit-only')

validation = run_validation(
    parse_args(arguments),
    persist_analysis=False,
    fast_cache=True,
)

status = validation.component_status.groupby(['seed_group', 'status']).size().unstack(fill_value=0)
print('Component cache:')
print(status.to_string())
print('Complete seed groups used:', sorted(validation.complete_grouped))
if not validation.complete_grouped:
    print('SMC estimate unavailable: complete Phase B before the final comparison.')
else:
    assert sorted(validation.complete_grouped) == SEED_GROUPS
    assert len(validation.specs) == len(COMPONENTS)
    print('All official-public SMC components are complete and exactly cache-compatible.')

In [ ]:
# 6. F_j estimators, variance, and comparison with IID direct sampling.
import math
import numpy as np
import pandas as pd
from IPython.display import display
from rea_llms.statistics import boxplot_statistics, weighted_quantile

direct_stats = direct_summary['statistics']
with np.load(DIRECT_ROOT / direct_summary['ari_file']) as archive:
    direct_ari_values = np.asarray(archive['raw_ari'], dtype=float)
direct_tails = {float(row['threshold']): row for row in direct_stats['tails']}
aggregate = validation.comparison.query("row_type == 'aggregate'").copy()
aggregate_by_threshold = {float(row.threshold): row for _, row in aggregate.iterrows()}

result_rows = []
for threshold in map(float, TAIL_THRESHOLDS):
    direct = direct_tails[threshold]
    smc = aggregate_by_threshold.get(threshold)
    p_direct = float(direct['empirical_probability'])
    p_smc = math.nan if smc is None else float(smc.mixture_probability)
    result_rows.append({
        'threshold': threshold,
        'direct events': int(direct['count']),
        'direct probability': p_direct,
        'direct variance': p_direct * (1.0 - p_direct) / int(direct_stats['count']),
        'SMC-SNIS mean': p_smc,
        'between-seed variance': math.nan if smc is None or pd.isna(smc.mixture_probability_std)
            else float(smc.mixture_probability_std) ** 2,
        'relative error': math.nan if not math.isfinite(p_smc) or p_direct == 0
            else (p_smc - p_direct) / p_direct,
        'mean corrected ESS/N': math.nan if smc is None else float(smc.weight_ess_over_n),
    })
results = pd.DataFrame(result_rows)
display(results.style.format({
    'threshold': '{:g}', 'direct events': '{:,}',
    'direct probability': '{:.7g}', 'direct variance': '{:.3g}',
    'SMC-SNIS mean': '{:.7g}', 'between-seed variance': '{:.3g}',
    'relative error': '{:+.1%}', 'mean corrected ESS/N': '{:.3f}',
}, na_rep='requires >=2 complete seed groups'))

# Every row below is one separately normalized F_j(a), never a pooled weight.
fj = validation.component_diagnostics[[
    'seed_group', 'cap', 'lambda_final', 'threshold',
    'component_probability', 'corrected_ess_over_n', 'raw_event_count',
    'log_z_smc', 'log_z_reverse', 'delta_log_z_smc_minus_reverse',
]].copy()
print('Individual component estimators F_j(a):')
display(fj.sort_values(['threshold', 'cap', 'lambda_final', 'seed_group']))

component_rows = []
component_populations = {}
for seed_group, components in validation.complete_grouped.items():
    for component in components:
        key = (float(component.spec.cap), float(component.spec.lambda_final))
        values = np.asarray(component.raw_ari, dtype=float)
        log_weights = np.asarray(component.final_log_weights, dtype=float)
        weights = np.exp(log_weights - np.logaddexp.reduce(log_weights))
        component_populations.setdefault(key, []).append((values, weights))
        median, q75 = weighted_quantile(values, weights, [.5, .75])
        component_rows.append({
            'seed_group': seed_group, 'cap': key[0], 'lambda': key[1],
            'weighted mean raw ARI': np.sum(weights * values),
            'weighted median raw ARI': median, 'weighted q75 raw ARI': q75,
            'weighted fraction at/above cap': np.sum(weights * (values >= key[0])),
            'terminal ESS/N': 1.0 / (len(weights) * np.sum(weights ** 2)),
            'runtime (s)': component.runtime_seconds,
        })
tilting = pd.DataFrame(component_rows)
if not tilting.empty:
    print('Tilting effectiveness by terminal component:')
    display(tilting.groupby(['cap', 'lambda']).agg(
        mean_raw_ari=('weighted mean raw ARI', 'mean'),
        q75_raw_ari=('weighted q75 raw ARI', 'mean'),
        fraction_at_or_above_cap=('weighted fraction at/above cap', 'mean'),
        terminal_ess_over_n=('terminal ESS/N', 'mean'),
        between_seed_variance=('weighted mean raw ARI', 'var'),
        mean_runtime_seconds=('runtime (s)', 'mean'),
    ).reset_index())

seed_estimates = validation.comparison.query("row_type == 'seed_group'").copy()
print('Complete seed groups used:', sorted(seed_estimates.seed_group.dropna().unique()))
if seed_estimates.seed_group.nunique() < 2:
    print('Between-seed variance is not estimable yet; run at least two complete seed groups.')


In [ ]:
# 7. Probability error, F_j variance, tilting, and tilted-distribution boxplots.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
x = results['threshold'].to_numpy(float)

ax = axes[0, 0]
ax.plot(x, results['direct probability'], marker='s', label='IID direct')
if results['SMC-SNIS mean'].notna().all():
    ax.plot(x, results['SMC-SNIS mean'], marker='^', label='mean of F_j')
ax.set_yscale('log'); ax.set_xticks(x)
ax.set_title('Base-model tail probabilities'); ax.set_xlabel('raw ARI threshold')
ax.set_ylabel('probability'); ax.grid(alpha=.25, which='both'); ax.legend()

ax = axes[0, 1]
if results['relative error'].notna().all():
    ax.bar(x.astype(int).astype(str), 100 * results['relative error'])
    ax.axhline(0, color='black', linewidth=1)
ax.set_title('Relative error vs IID direct'); ax.set_xlabel('raw ARI threshold')
ax.set_ylabel('relative error (%)'); ax.grid(axis='y', alpha=.25)

ax = axes[1, 0]
if component_populations:
    keys = sorted(component_populations)
    direct_box = boxplot_statistics(direct_ari_values)
    bxp_rows = [{
        'label': 'IID direct', 'q1': direct_box['q1'], 'med': direct_box['median'],
        'q3': direct_box['q3'], 'whislo': direct_box['whisker_low'],
        'whishi': direct_box['whisker_high'], 'fliers': [],
    }]
    for cap, lam in keys:
        populations = component_populations[(cap, lam)]
        values = np.concatenate([values for values, _ in populations])
        weights = np.concatenate([weights / len(populations) for _, weights in populations])
        stats = boxplot_statistics(values, weights=weights)
        bxp_rows.append({
            'label': f'C={cap:g}\nl={lam:g}', 'q1': stats['q1'],
            'med': stats['median'], 'q3': stats['q3'],
            'whislo': stats['whisker_low'], 'whishi': stats['whisker_high'],
            'fliers': [],
        })
    ax.bxp(bxp_rows, showfliers=False)
    ax.tick_params(axis='x', labelrotation=45)
ax.set_title('Final tilted SMC populations'); ax.set_ylabel('raw ARI')

ax = axes[1, 1]
if not tilting.empty:
    grouped = tilting.groupby(['cap', 'lambda'], as_index=False)['weighted mean raw ARI'].mean()
    for cap, rows in grouped.groupby('cap'):
        ax.plot(rows['lambda'], rows['weighted mean raw ARI'], marker='o', label=f'cap={cap:g}')
    ax.legend()
ax.set_title('Does stronger tilting move ARI?'); ax.set_xlabel('terminal lambda')
ax.set_ylabel('mean raw ARI'); ax.grid(alpha=.25)

fig.tight_layout()
plt.show()

if not fj.empty:
    fig, axes = plt.subplots(1, len(TAIL_THRESHOLDS), figsize=(5 * len(TAIL_THRESHOLDS), 4), squeeze=False)
    for ax, threshold in zip(axes[0], map(float, TAIL_THRESHOLDS)):
        rows = fj[fj.threshold == threshold]
        labels = [f'C={c:g}, l={l:g}' for c, l in zip(rows.cap, rows.lambda_final)]
        ax.scatter(range(len(rows)), rows.component_probability, c=rows.corrected_ess_over_n, cmap='viridis')
        ax.axhline(direct_tails[threshold]['empirical_probability'], color='black', linestyle='--', label='direct')
        ax.set_xticks(range(len(rows)), labels, rotation=90)
        ax.set_yscale('log'); ax.set_title(f'F_j for ARI >= {threshold:g}')
        ax.set_ylabel('probability estimate'); ax.legend()
    fig.tight_layout(); plt.show()

print('SMC boxplots and tilting summaries use terminal weights W_j.')
print('F_j reverse correction and component averaging are shown separately.')
